# 06 — Model Training & Evaluation

**Goal:** Train machine learning models to predict blood pressure from PPG features and evaluate their performance.

**Six BP targets:**

| Variable | Meaning |
|----------|---------|
| `MAP`    | Mean Arterial Pressure (absolute, mmHg) |
| `DMAP`   | Change in MAP from baseline |
| `SBP`    | Systolic Blood Pressure |
| `DSBP`   | Change in SBP |
| `DBP`    | Diastolic Blood Pressure |
| `DDBP`   | Change in DBP |

**Models tested (4 hyperparameter grids):**
- SVR — RBF kernel
- SVR — Polynomial kernel
- Decision Tree
- Random Forest

All models are evaluated on R², RMSE, and MAE.
The AAMI clinical standard requires RMSE ≤ 5 mmHg.

**Inputs:** `data/processed/features_train.pkl`, `data/processed/features_test.pkl`

**Outputs:** `outputs/models/` (`.sav` files), `outputs/results/` (CSVs + plots)

---
### Pipeline position
```
01 Import → 02 Signal Processing → 03 Interpolation → 04 ML Data Loading → 05 Features → [06 Models]
```

In [ ]:
import time
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from joblib import dump, load

from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ROOT     = Path.cwd().parent if Path.cwd().name == 'improved' else Path.cwd().parent.parent
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
MODELS_DIR    = REPO_ROOT / 'outputs' / 'models'
RESULTS_DIR   = REPO_ROOT / 'outputs' / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Cross-validation settings
CV_N_SPLITS  = 10
CV_TEST_SIZE = 0.20
RANDOM_STATE = 42

# AAMI clinical accuracy standard
AAMI_RMSE_THRESHOLD = 5.0   # mmHg

print(f'Models output : {MODELS_DIR}')
print(f'Results output: {RESULTS_DIR}')

## 1. Load Data

In [ ]:
with open(PROCESSED_DIR / 'features_train.pkl', 'rb') as f:
    features_train = pickle.load(f)
with open(PROCESSED_DIR / 'features_test.pkl', 'rb') as f:
    features_test = pickle.load(f)

print(f'Train: {len(features_train)} windows,  {features_train.shape[1]} columns')
print(f'Test : {len(features_test)}  windows,  {features_test.shape[1]} columns')

## 2. Split Features and Labels

Columns that are BP labels are separated from feature columns.
Patient IDs are kept for grouped cross-validation (so no patient leaks between train and validation folds).

In [ ]:
# BP target column names — must match what notebook 05 saved
LABEL_COLS = ['MAP', 'DMAP', 'SBP', 'DSBP', 'DBP', 'DDBP']

def split_X_Y(df: pd.DataFrame) -> tuple:
    """
    Separate feature matrix, label DataFrame, and patient IDs.

    Returns
    -------
    X       : feature DataFrame (all columns except patient_id and label columns)
    Y       : label DataFrame (MAP, DMAP, SBP, DSBP, DBP, DDBP)
    subjects: Series of patient IDs (for grouped cross-validation)
    """
    existing_labels = [c for c in LABEL_COLS if c in df.columns]
    drop_cols = ['patient_id'] + existing_labels

    X        = df.drop(columns=drop_cols, errors='ignore')
    Y        = df[existing_labels]
    subjects = df['patient_id'] if 'patient_id' in df.columns else pd.Series(range(len(df)))
    return X, Y, subjects


X_train, Y_train, subjects_train = split_X_Y(features_train)
X_test,  Y_test,  subjects_test  = split_X_Y(features_test)

print(f'Feature matrix : X_train {X_train.shape},  X_test {X_test.shape}')
print(f'Label matrix   : Y_train {Y_train.shape},  Y_test {Y_test.shape}')
print(f'Labels present : {Y_train.columns.tolist()}')

## 3. ML Pipeline

All models share the same preprocessing pipeline:
1. **Impute** missing feature values with column means
2. **Scale** features to zero-mean, unit-variance
3. **PCA** — keep components explaining 95% of variance (reduces 356 features, removes collinearity)
4. **Estimator** — the model being evaluated

Grouped cross-validation (`GroupShuffleSplit`) ensures no patient appears in both train and validation within the same fold.

In [ ]:
def make_pipeline(estimator) -> Pipeline:
    """Build the standard preprocessing + estimator pipeline."""
    return Pipeline([
        ('imputer',   SimpleImputer(strategy='mean')),
        ('scaler',    StandardScaler()),
        ('pca',       PCA(n_components=0.95)),
        ('estimator', estimator),
    ])


# ── Hyperparameter grids ────────────────────────────────────────────────────────
GRIDS = {
    'SVR_RBF': (
        make_pipeline(SVR(kernel='rbf')),
        {
            'estimator__C':     [0.1, 1, 10, 100],
            'estimator__gamma': ['scale', 'auto'],
            'pca__n_components': [0.90, 0.95, 0.99],
        }
    ),
    'SVR_Poly': (
        make_pipeline(SVR(kernel='poly')),
        {
            'estimator__C':      [0.1, 1, 10],
            'estimator__degree': [2, 3, 4],
            'estimator__gamma':  ['scale', 'auto'],
        }
    ),
    'DecisionTree': (
        make_pipeline(DecisionTreeRegressor(random_state=RANDOM_STATE)),
        {
            'estimator__max_depth':        [5, 10, 15, 20],
            'estimator__min_samples_split': [2, 5, 10],
            'estimator__criterion':         ['squared_error', 'absolute_error'],
        }
    ),
    'RandomForest': (
        make_pipeline(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        {
            'estimator__n_estimators': [50, 100, 200],
            'estimator__max_depth':    [10, 20, 30],
            'estimator__min_samples_split': [2, 5],
        }
    ),
}

## 4. Training Loop

For each combination of (target label × hyperparameter grid), run a grouped GridSearchCV and record the best model + metrics.

In [ ]:
def train_and_evaluate(X_train, X_test, y_train, y_test,
                        subjects, pipe, param_grid,
                        label_name: str, grid_name: str) -> dict:
    """
    Run GridSearchCV, evaluate on the test set, save model and results.

    Parameters
    ----------
    X_train, X_test   : feature DataFrames
    y_train, y_test   : label Series
    subjects          : patient ID Series for grouped CV
    pipe              : sklearn Pipeline
    param_grid        : hyperparameter search space
    label_name        : e.g. 'MAP', 'SBP'
    grid_name         : e.g. 'SVR_RBF'

    Returns
    -------
    dict with best_params, r2, rmse, mae, aami_pass flag
    """
    t0 = time.time()

    cv = GroupShuffleSplit(n_splits=CV_N_SPLITS, test_size=CV_TEST_SIZE,
                           random_state=RANDOM_STATE)
    gs = GridSearchCV(pipe, param_grid,
                      cv=cv.split(X_train, groups=subjects),
                      scoring='r2', n_jobs=-1, refit=True,
                      return_train_score=True)
    gs.fit(X_train, y_train)

    y_pred = gs.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    elapsed = time.time() - t0
    aami_pass = rmse <= AAMI_RMSE_THRESHOLD

    print(f'  {label_name} | {grid_name:<14} '
          f'R²={r2:.3f}  RMSE={rmse:.2f}  MAE={mae:.2f}  '
          f'[{"PASS" if aami_pass else "FAIL"} AAMI]  ({elapsed:.0f}s)')

    # Save model
    model_path = MODELS_DIR / f'{label_name}_{grid_name}_model.sav'
    dump(gs.best_estimator_, model_path)

    # Save CV results
    cv_results = pd.DataFrame(gs.cv_results_)
    cv_results.to_csv(RESULTS_DIR / f'{label_name}_{grid_name}_cv_results.csv', index=False)

    # Prediction plot
    _plot_predictions(y_test, y_pred, label_name, grid_name)

    return {
        'label':       label_name,
        'grid':        grid_name,
        'best_params': gs.best_params_,
        'r2':          r2,
        'rmse':        rmse,
        'mae':         mae,
        'aami_pass':   aami_pass,
    }


def _plot_predictions(y_true, y_pred, label_name: str, grid_name: str):
    """Save a true-vs-predicted scatter plot."""
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_true, y_pred, alpha=0.4, s=15, color='steelblue')
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
    ax.set_xlabel('True BP (mmHg)')
    ax.set_ylabel('Predicted BP (mmHg)')
    ax.set_title(f'{label_name} — {grid_name}  (R²={r2_score(y_true, y_pred):.3f})')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'{label_name}_{grid_name}_scatter.png', dpi=120)
    plt.show()
    plt.close()

In [ ]:
# ── Run all label × grid combinations ─────────────────────────────────────────
all_results = []
existing_labels = [c for c in LABEL_COLS if c in Y_train.columns]

for label in existing_labels:
    print(f'\n── {label} ─────────────────────────────────')
    y_train_label = Y_train[label].dropna()
    y_test_label  = Y_test[label].dropna()
    X_tr = X_train.loc[y_train_label.index]
    X_te = X_test.loc[y_test_label.index]
    subs = subjects_train.loc[y_train_label.index]

    for grid_name, (pipe, param_grid) in GRIDS.items():
        result = train_and_evaluate(
            X_tr, X_te, y_train_label, y_test_label,
            subs, pipe, param_grid, label, grid_name
        )
        all_results.append(result)

print('\nAll models trained.')

## 5. Baseline — Linear Regression on Raw Signal

As a sanity check, a simple linear regression is trained on downsampled raw PPG waveforms (no features). This measures how much of the predictive power comes from the signal shape alone vs. the engineered features.

In [ ]:
# Load raw PPG signals if available
try:
    with open(PROCESSED_DIR / 'ml_train.pkl', 'rb') as f:
        ml_train = pickle.load(f)
    with open(PROCESSED_DIR / 'ml_test.pkl', 'rb') as f:
        ml_test = pickle.load(f)

    # Downsample from 6000 → 600 samples (10x) to reduce dimensionality
    DOWNSAMPLE_FACTOR = 10

    def build_raw_matrix(ppg_dict, patient_ids, window_samples=6000):
        rows, pids = [], []
        for pid in patient_ids:
            if pid not in ppg_dict:
                continue
            sig = ppg_dict[pid]['PLETH'].values
            for start in range(0, len(sig) - window_samples + 1, window_samples):
                chunk = sig[start: start + window_samples]
                if np.isnan(chunk).mean() < 0.1:
                    rows.append(chunk[::DOWNSAMPLE_FACTOR])
                    pids.append(pid)
        return pd.DataFrame(rows), pd.Series(pids)

    X_raw_train, _ = build_raw_matrix(ml_train['ppg'], ml_train['patient_ids'])
    X_raw_test,  _ = build_raw_matrix(ml_test['ppg'],  ml_test['patient_ids'])

    # Align lengths with feature DataFrames
    min_len_tr = min(len(X_raw_train), len(Y_train))
    min_len_te = min(len(X_raw_test),  len(Y_test))

    print(f'Raw signal baseline matrix: train {X_raw_train.shape}, test {X_raw_test.shape}')
    Y_names = list(Y_train.columns)

    baseline_results = []
    lr = LinearRegression()
    for label in existing_labels:
        y_tr = Y_train[label].iloc[:min_len_tr].values
        y_te = Y_test[label].iloc[:min_len_te].values
        X_tr = X_raw_train.iloc[:min_len_tr].values
        X_te = X_raw_test.iloc[:min_len_te].values

        lr.fit(X_tr, y_tr)
        y_pred = lr.predict(X_te)
        r2   = r2_score(y_te, y_pred)
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        print(f'  LinearReg baseline — {label}: R²={r2:.3f}  RMSE={rmse:.2f}')
        baseline_results.append({'label': label, 'grid': 'LinearReg_baseline', 'r2': r2, 'rmse': rmse})

except FileNotFoundError:
    print('ml_train.pkl / ml_test.pkl not found — skipping raw-signal baseline.')

## 6. Results Summary

In [ ]:
results_df = pd.DataFrame(all_results)
results_df['rmse'] = results_df['rmse'].round(3)
results_df['r2']   = results_df['r2'].round(3)
results_df['mae']  = results_df['mae'].round(3)

print('\n── Full Results ───────────────────────────────────────────────────────')
print(results_df[['label', 'grid', 'r2', 'rmse', 'mae', 'aami_pass']].to_string(index=False))

# Best model per label
print('\n── Best model per label (highest R²) ──────────────────────────────────')
best = results_df.loc[results_df.groupby('label')['r2'].idxmax()]
print(best[['label', 'grid', 'r2', 'rmse', 'aami_pass']].to_string(index=False))

results_df.to_csv(RESULTS_DIR / 'all_results_summary.csv', index=False)
print('\nFull results saved → outputs/results/all_results_summary.csv')

In [ ]:
# ── RMSE comparison plot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
pivot = results_df.pivot(index='label', columns='grid', values='rmse')
pivot.plot(kind='bar', ax=ax, width=0.7)
ax.axhline(AAMI_RMSE_THRESHOLD, color='red', linestyle='--',
           linewidth=1.5, label=f'AAMI threshold ({AAMI_RMSE_THRESHOLD} mmHg)')
ax.set_title('RMSE by Target and Model')
ax.set_xlabel('BP Target')
ax.set_ylabel('RMSE (mmHg)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'rmse_comparison.png', dpi=120)
plt.show()